# 43. Hallucinations, reasoning checks, and LLM evaluation

**Level:** Advanced

**Prerequisites:** 20, 40-42: metrics, boundaries, and retrieval.

[Course index](00_START_HERE.ipynb) | [Previous](42_Retrieval_Augmented_Generation.ipynb) | [Next](44_FineTuning_LoRA_Distillation_and_Preferences.ipynb)

**Topics:** Hallucination, Factual reliability, Reasoning, LLM evaluation, Groundedness, Regression tests.

**Matching script:** `examples\43_LLM_Evaluation_Hallucinations_and_Reliability.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A hallucination is an unsupported or incorrect generated claim presented as if reliable.
Fluent language, a citation-shaped string, high confidence, or low temperature do not establish
factual support. A model can also fail by omitting a required caveat, executing an unauthorized
tool, or producing valid JSON with the wrong values.

Separate evaluation dimensions: correctness, groundedness, format, tool behavior, latency, cost,
privacy, and safe abstention. Use representative held-out cases with normal, edge, adversarial,
and missing-evidence inputs. Keep versions of prompts, model IDs, datasets, and scoring rules.

Exact match is useful for constrained outputs but overly strict for paraphrased answers.
Semantic judges can add coverage but introduce their own errors, bias, and nondeterminism.
Calibrate judges against human labels; do not let the same system judge itself uncritically.
Check final answers and externally verifiable outcomes instead of assuming a long explanation
is a faithful proof of reasoning.

The exercise scores fixed mock outputs to teach an evaluation harness. It does not claim to
measure a real model or guarantee that lexical citation checks establish entailment.

## Mathematics or data contract

Task pass rate = passing independent cases / evaluated cases; report the case distribution and uncertainty.

## Important limitation

A two-case deterministic harness verifies its own scoring logic only.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Evaluate correctness and evidence separately

**Purpose:** Catch valid-looking but unsupported output.

### Read the code in this order
1. Declare expected values and supporting source IDs.
2. Score mock outputs against both dimensions.
3. Keep per-case failure reasons.
4. Assert the fabricated value fails even though it has JSON structure.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
per-case evaluation: [{'id': 'known', 'correct': False, 'grounded': False, 'passed': False}, {'id': 'unknown', 'correct': True, 'grounded': True, 'passed': True}]
mock-output pass rate: 0.5
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
cases = [
    {"id": "known", "expected": "1500", "allowed_source": "mtu-policy"},
    {"id": "unknown", "expected": "unknown", "allowed_source": None},
]
mock_outputs = {
    "known": {"answer": "9000", "source": "made-up"},
    "unknown": {"answer": "unknown", "source": None},
}
results = []
for case in cases:
    output = mock_outputs[case["id"]]
    correct = output["answer"] == case["expected"]
    grounded = output["source"] == case["allowed_source"]
    results.append({"id": case["id"], "correct": correct, "grounded": grounded,
                    "passed": correct and grounded})
assert results[0]["passed"] is False and results[1]["passed"] is True
pass_rate = sum(item["passed"] for item in results) / len(results)
assert pass_rate == 0.5
print("per-case evaluation:", results)
print("mock-output pass rate:", pass_rate)

per-case evaluation: [{'id': 'known', 'correct': False, 'grounded': False, 'passed': False}, {'id': 'unknown', 'correct': True, 'grounded': True, 'passed': True}]
mock-output pass rate: 0.5


### Interpret example 1

The result should demonstrate this purpose: **Catch valid-looking but unsupported output.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add a correct answer with a wrong source and a wrong answer with a correct source; score these separately.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Hallucination` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Task pass rate = passing independent cases / evaluated cases; report the case distribution and uncertainty.
- Name the boundary of the demonstration: A two-case deterministic harness verifies its own scoring logic only.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.